
# Event → Market Impact: Oils, Tech, Energy, S&P
A structured, testable workflow to learn how factors & events affect **SPY, XLK, XLE** (and optionally individual tickers) and to query **what-if** scenarios.

**Core ideas**
- Predict **next-day returns** (not raw prices) to avoid scale issues and focus on impact.
- Use a **time-series cross-validation** setup that avoids lookahead bias.
- Build interpretable drivers (macro, commodities, FX, sentiment, trends, news) with proper lags & frequency alignment.
- Provide a **what-if API**: input a shock (e.g., `VIX +10%`, `Gold -5%`, `Fed Funds +25 bps`) → get predicted returns.

We'll start simple and layer complexity step-by-step.



## 1) Environment & packages
> If you're running locally and miss some packages, uncomment the `pip` installs below.


In [1]:

# OPTIONAL: uncomment if needed (and your environment allows installs)
# %pip install --upgrade yfinance pandas_datareader pytrends feedparser nltk sentence-transformers scikit-learn shap matplotlib
# %pip install transformers torch --index-url https://download.pytorch.org/whl/cu121  # or cpu build if needed

import os, re, json, math, warnings
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (9, 5)

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)



## 2) Project configuration
Edit tickers, date range, and data sources here. By default we model **SPY** (S&P 500), **XLK** (Tech), **XLE** (Energy).  
Drivers include Oil (CL=F), Gold (GC=F), VIX (^VIX), DXY (DX-Y.NYB), EURUSD, USDJPY, plus macro (FRED) and Google Trends.


In [2]:

# ---- Target universe (you can add/remove tickers) ----
TARGETS = ["SPY", "XLK", "XLE"]  # S&P 500, Tech sector ETF, Energy sector ETF

# ---- Driver series via yfinance (daily) ----
YF_SERIES = {
    "oil": "CL=F",
    "gold": "GC=F",
    "vix": "^VIX",
    "dxy": "DX-Y.NYB",
    "eurusd": "EURUSD=X",
    "usdjpy": "JPY=X",
}

# ---- Macro from FRED (low-frequency; will be forward-filled) ----
FRED_SERIES = {
    "cpi": "CPIAUCSL",
    "unemployment": "UNRATE",
    "teny": "DGS10",
    "fedfunds": "FEDFUNDS",
}

# ---- Google Trends (optional) ----
TRENDS_KEYWORDS = [
    "recession", "inflation", "stock market", "oil prices", "interest rates"
]

# ---- RSS feeds for simple news headline sentiment (optional) ----
RSS_FEEDS = [
    "http://feeds.reuters.com/reuters/businessNews",
    "http://feeds.reuters.com/reuters/USMarketNews",
]

# ---- Time window ----
YEARS_BACK = 5
END = datetime.today()
START = END - timedelta(days=365 * YEARS_BACK)

# ---- Data & cache paths ----
DATA_DIR = Path("./data_cache")
DATA_DIR.mkdir(parents=True, exist_ok=True)



## 3) Data loaders (with lightweight on-disk cache)
All fetchers return **pd.Series/DataFrames** indexed by datetime. If a source is unavailable, they return empty objects and the pipeline continues.


In [3]:

def _cache_path(name: str) -> Path:
    safe = re.sub(r"[^a-zA-Z0-9_.-]", "_", name)
    return DATA_DIR / f"{safe}.parquet"

def _to_parquet(df: pd.DataFrame, path: Path):
    try:
        df.to_parquet(path)
    except Exception:
        # Parquet engine might be missing; fallback to CSV
        df.to_csv(path.with_suffix(".csv"))

def _from_parquet(path: Path) -> pd.DataFrame:
    try:
        return pd.read_parquet(path)
    except Exception:
        return pd.read_csv(path.with_suffix(".csv"), index_col=0, parse_dates=True)

def _read_cache(name: str):
    p = _cache_path(name)
    if p.exists() or p.with_suffix(".csv").exists():
        return _from_parquet(p)
    return None

def _write_cache(name: str, df: pd.DataFrame):
    p = _cache_path(name)
    _to_parquet(df, p)


In [4]:

# Optional imports inside functions to keep the notebook light
def fetch_yf_close(ticker: str, start: datetime, end: datetime) -> pd.Series:
    import yfinance as yf
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if "Close" in df.columns:
        s = df["Close"].dropna().rename(ticker)
        s.index = pd.to_datetime(s.index).tz_localize(None)
        return s
    return pd.Series(dtype=float, name=ticker)

def fetch_targets(targets, start, end) -> pd.DataFrame:
    key = f"targets_{'_'.join(targets)}_{start.date()}_{end.date()}"
    cached = _read_cache(key)
    if cached is not None:
        return cached
    frames = []
    for t in targets:
        try:
            frames.append(fetch_yf_close(t, start, end))
        except Exception as e:
            print(f"[WARN] target {t}: {e}")
    df = pd.concat(frames, axis=1).dropna(how='all')
    _write_cache(key, df)
    return df

def fetch_yf_drivers(series_map, start, end) -> pd.DataFrame:
    key = f"yfdrivers_{start.date()}_{end.date()}"
    cached = _read_cache(key)
    if cached is not None:
        return cached
    frames = []
    for name, ticker in series_map.items():
        try:
            s = fetch_yf_close(ticker, start, end).rename(name)
            frames.append(s)
        except Exception as e:
            print(f"[WARN] yfinance driver {name} ({ticker}): {e}")
    df = pd.concat(frames, axis=1).dropna(how='all')
    _write_cache(key, df)
    return df

def fetch_fred(series_map, start, end) -> pd.DataFrame:
    key = f"fred_{start.date()}_{end.date()}"
    cached = _read_cache(key)
    if cached is not None:
        return cached
    try:
        from pandas_datareader import data as pdr
    except Exception:
        print("[INFO] pandas_datareader missing; skipping FRED.")
        return pd.DataFrame()
    frames = []
    for name, sid in series_map.items():
        try:
            s = pdr.DataReader(sid, "fred", start, end).squeeze().rename(name)
            s.index = pd.to_datetime(s.index).tz_localize(None)
            frames.append(s)
        except Exception as e:
            print(f"[WARN] FRED {name} ({sid}): {e}")
    df = pd.concat(frames, axis=1).dropna(how='all')
    _write_cache(key, df)
    return df

def fetch_trends(keywords, start, end, geo="US") -> pd.DataFrame:
    key = f"trends_{geo}_{start.date()}_{end.date()}"
    cached = _read_cache(key)
    if cached is not None:
        return cached
    try:
        from pytrends.request import TrendReq
        tr = TrendReq(hl="en-US", tz=360)
        tr.build_payload(keywords, timeframe=f"{start.strftime('%Y-%m-%d')} {end.strftime('%Y-%m-%d')}", geo=geo)
        df = tr.interest_over_time()
        if "isPartial" in df.columns:
            df = df.drop(columns=["isPartial"])
        df.index = pd.to_datetime(df.index).tz_localize(None)
        _write_cache(key, df)
        return df
    except Exception as e:
        print(f"[INFO] Google Trends unavailable: {e}")
        return pd.DataFrame()


In [5]:

def fetch_news_rss(feeds) -> pd.DataFrame:
    try:
        import feedparser
    except Exception:
        print("[INFO] feedparser missing; skipping RSS.")
        return pd.DataFrame(columns=["published", "source", "headline"])
    rows = []
    for url in feeds:
        try:
            parsed = feedparser.parse(url)
            for entry in parsed.entries:
                published = None
                if hasattr(entry, "published_parsed") and entry.published_parsed:
                    published = datetime(*entry.published_parsed[:6])
                elif hasattr(entry, "updated_parsed") and entry.updated_parsed:
                    published = datetime(*entry.updated_parsed[:6])
                if published is None: 
                    continue
                headline = entry.title if hasattr(entry, "title") else ""
                rows.append({"published": published, "source": parsed.href if hasattr(parsed, "href") else "rss", "headline": headline})
        except Exception as e:
            print(f"[WARN] RSS {url}: {e}")
    return pd.DataFrame(rows)



## 4) Headline sentiment (optional)
We compute **VADER** and **FinBERT** sentiment on headlines and aggregate daily.


In [6]:

def clean_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"https?://\S+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    return " ".join(w for w in s.split() if len(w) > 2)

def score_vader(texts):
    try:
        import nltk
        from nltk.sentiment import SentimentIntensityAnalyzer
        # ensure lexicon
        try:
            nltk.data.find('sentiment/vader_lexicon.zip')
        except LookupError:
            nltk.download('vader_lexicon')
        sia = SentimentIntensityAnalyzer()
        return np.array([sia.polarity_scores(t).get("compound", np.nan) for t in texts])
    except Exception:
        return np.array([np.nan]*len(texts))

def score_finbert(texts):
    try:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        import torch
        tok = AutoTokenizer.from_pretrained("ProsusAI/finbert")
        mdl = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
        scores = []
        for t in texts:
            inputs = tok(t, return_tensors="pt", truncation=True, max_length=128)
            with torch.no_grad():
                logits = mdl(**inputs).logits
            probs = torch.softmax(logits, dim=1).numpy()[0]
            scores.append(float(probs[2] - probs[0]))  # pos - neg
        return np.array(scores)
    except Exception:
        return np.array([np.nan]*len(texts))

def aggregate_headline_sentiment(df_text: pd.DataFrame, freq='D') -> pd.DataFrame:
    if df_text is None or df_text.empty:
        return pd.DataFrame()
    df = df_text.copy()
    df["clean"] = df["headline"].astype(str).map(clean_text)
    df["vader"] = score_vader(df["clean"].tolist())
    df["finbert"] = score_finbert(df["clean"].tolist())
    df["date"] = pd.to_datetime(df["published"]).tz_localize(None).to_series().dt.floor(freq)
    agg = df.groupby("date").agg(news_count=("headline", "count"),
                                 vader_mean=("vader", "mean"),
                                 finbert_mean=("finbert", "mean"))
    return agg



## 5) Assemble unified dataset
- Align everything to **business days**.
- Convert prices to **returns** (`pct_change`).
- Build **lagged** features (no lookahead!).
- Add simple technicals (rolling mean/std, momentum) and calendar features.


In [7]:

def to_business_daily(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    return df.asfreq('B').ffill()

def pct_returns(df: pd.DataFrame, add_suffix=True) -> pd.DataFrame:
    ret = df.pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if add_suffix:
        ret.columns = [f"{c}_ret" for c in ret.columns]
    return ret

def add_calendar_features(index: pd.DatetimeIndex) -> pd.DataFrame:
    cal = pd.DataFrame(index=index)
    cal["dow"] = index.weekday
    cal["month"] = index.month
    # one-hot safely
    cal = pd.get_dummies(cal, columns=["dow", "month"], drop_first=True)
    return cal

def rolling_features(df: pd.DataFrame, windows=(5, 10, 21)) -> pd.DataFrame:
    feats = pd.DataFrame(index=df.index)
    for col in df.columns:
        s = df[col]
        for w in windows:
            feats[f"{col}_roll_mean_{w}"] = s.rolling(w).mean()
            feats[f"{col}_roll_std_{w}"] = s.rolling(w).std()
            feats[f"{col}_mom_{w}"] = s.pct_change(w)
    return feats

def lagged(df: pd.DataFrame, lags=(1,2,3,5,10)) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    for col in df.columns:
        for L in lags:
            out[f"{col}_lag_{L}"] = df[col].shift(L)
    return out

def assemble_dataset(start=START, end=END):
    # Targets (prices → returns)
    prices = fetch_targets(TARGETS, start, end)
    prices = to_business_daily(prices)
    t_rets = pct_returns(prices, add_suffix=False)  # columns: SPY, XLK, XLE (returns)
    # Drivers (prices → returns); Macro → ffill; Trends → as-is
    yfdrv = to_business_daily(fetch_yf_drivers(YF_SERIES, start, end))
    yf_rets = pct_returns(yfdrv)
    fred = to_business_daily(fetch_fred(FRED_SERIES, start, end))
    trends = fetch_trends(TRENDS_KEYWORDS, start, end)
    news = aggregate_headline_sentiment(fetch_news_rss(RSS_FEEDS))
    # Merge
    df = pd.concat([t_rets, yf_rets, fred, trends, news], axis=1)
    df = df.sort_index().ffill()
    # Feature matrix X and target y (next-day returns)
    # Build features ONLY from drivers & past target information (no leakage from future targets)
    driver_cols = [c for c in df.columns if c not in TARGETS]  # exclude target return columns
    X_raw = pd.concat([
        lagged(df[driver_cols]),
        rolling_features(df[driver_cols]),
        add_calendar_features(df.index),
    ], axis=1)
    # Optionally include own-lag returns of targets as predictors
    X_raw = pd.concat([X_raw, lagged(df[TARGETS])], axis=1)
    # Target is next-day return for each target
    y = df[TARGETS].shift(-1)
    # Align & clean
    data = pd.concat([X_raw, y], axis=1).dropna()
    X = data[X_raw.columns]
    Y = data[TARGETS]
    return X, Y, df

X, Y, raw_df = assemble_dataset()
print("Shapes:", X.shape, Y.shape)
X.tail(3)


Failed to get ticker 'SPY' reason: Failed to perform, curl: (35) TLS connect error: error:00000000:invalid library (0):OPENSSL_internal:invalid library (0). See https://curl.se/libcurl/c/libcurl-errors.html first for more details.

1 Failed download:
['SPY']: Timeout('Failed to perform, curl: (28) Connection timed out after 30002 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')


KeyboardInterrupt: 

Failed to get ticker 'XLK' reason: Failed to perform, curl: (28) Connection timed out after 10002 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.



## 6) Modeling & evaluation (time-series CV)
We'll run two baselines:
- **Ridge (linear)**
- **Random Forest**
We evaluate MAE and **directional accuracy**. You can plug in XGBoost/LightGBM later.


In [ ]:

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, accuracy_score

def directional_accuracy(y_true, y_pred):
    return np.mean(np.sign(y_true) == np.sign(y_pred))

def cv_backtest(X, Y, n_splits=5, gap=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    results = {t: {"ridge": [], "rf": []} for t in Y.columns}
    models_fitted = {t: {} for t in Y.columns}
    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        # Add a small gap between train/test to reduce leakage on rolling stats
        if gap > 0:
            test_idx = test_idx[gap:]
            if len(test_idx) == 0: 
                continue
        Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
        for t in Y.columns:
            ytr, yte = Y[t].iloc[train_idx], Y[t].iloc[test_idx]
            # Ridge
            ss = StandardScaler()
            Xtr_s = ss.fit_transform(Xtr)
            Xte_s = ss.transform(Xte)
            ridge = Ridge(alpha=1.0, random_state=RANDOM_SEED)
            ridge.fit(Xtr_s, ytr)
            pr = ridge.predict(Xte_s)
            mae = mean_absolute_error(yte, pr)
            da = directional_accuracy(yte.values, pr)
            results[t]["ridge"].append({"fold": fold, "mae": mae, "dir_acc": da})
            # RF
            rf = RandomForestRegressor(n_estimators=400, random_state=RANDOM_SEED, n_jobs=-1)
            rf.fit(Xtr, ytr)
            pr2 = rf.predict(Xte)
            mae2 = mean_absolute_error(yte, pr2)
            da2 = directional_accuracy(yte.values, pr2)
            results[t]["rf"].append({"fold": fold, "mae": mae2, "dir_acc": da2})
        # keep last fold models as provisional production models (per target)
        # retrain on all data later
    # Fit final models on full sample
    final_models = {}
    scalers = {}
    for t in Y.columns:
        ss = StandardScaler()
        Xs = ss.fit_transform(X)
        ridge = Ridge(alpha=1.0, random_state=RANDOM_SEED)
        ridge.fit(Xs, Y[t])
        rf = RandomForestRegressor(n_estimators=800, random_state=RANDOM_SEED, n_jobs=-1)
        rf.fit(X, Y[t])
        final_models[t] = {"ridge": ridge, "rf": rf}
        scalers[t] = ss
    return results, final_models, scalers

cv_results, models, scalers = cv_backtest(X, Y, n_splits=5, gap=5)

# Summaries
summary = []
for t in Y.columns:
    for mdl in ["ridge", "rf"]:
        mae = np.mean([r["mae"] for r in cv_results[t][mdl]]) if cv_results[t][mdl] else np.nan
        da = np.mean([r["dir_acc"] for r in cv_results[t][mdl]]) if cv_results[t][mdl] else np.nan
        summary.append({"target": t, "model": mdl, "cv_mae": mae, "cv_dir_acc": da})
pd.DataFrame(summary)



## 7) What drives predictions? (Permutation importance)
Use permutation importance on the **RandomForest** to estimate driver relevance. (Fast and robust; SHAP optional later.)


In [ ]:

from sklearn.inspection import permutation_importance

def perm_importance(model, X, y, n_repeats=10):
    r = permutation_importance(model, X, y, n_repeats=n_repeats, random_state=RANDOM_SEED, n_jobs=-1)
    imp = pd.DataFrame({"feature": X.columns, "importance": r.importances_mean})
    imp = imp.sort_values("importance", ascending=False)
    return imp

feature_ranks = {}
for t in Y.columns:
    imp = perm_importance(models[t]["rf"], X, Y[t])
    feature_ranks[t] = imp.head(40)  # top 40
feature_ranks[Y.columns[0]].head(20)



## 8) What-if API (factor shocks → predicted next-day returns)
Provide a dictionary of **percentage shocks** to latest driver returns (e.g., `{"oil_ret": +0.05, "vix_ret": +0.10}`) or **bps shocks** for rates (e.g., `{"fedfunds": 0.0025}`).


In [ ]:

def current_feature_row(X: pd.DataFrame) -> pd.Series:
    return X.iloc[-1].copy()

def apply_shocks_to_row(row: pd.Series, shocks: dict) -> pd.Series:
    new_row = row.copy()
    for key, val in shocks.items():
        # We expect keys to map to base driver names (oil, vix, gold, dxy, eurusd, usdjpy, fedfunds, etc.)
        # We will nudge the most recent lagged features:
        # e.g., 'oil_ret_lag_1', 'vix_ret_lag_1', or raw macro 'fedfunds_lag_1'
        # Heuristic mapping:
        prefixes = [
            f"{key}_ret_lag_1",
            f"{key}_lag_1",
        ]
        for pref in prefixes:
            if pref in new_row.index and pd.notna(new_row[pref]):
                new_row[pref] = new_row[pref] + val  # additive change to return or level
        # Optional: momentum windows
        for w in (5,10,21):
            mfeat = f"{key}_ret_mom_{w}"
            if mfeat in new_row.index and pd.notna(new_row[mfeat]):
                new_row[mfeat] = new_row[mfeat] + val
    return new_row

def predict_with_shocks(models, scalers, X, shocks: dict, use="rf") -> pd.Series:
    row = current_feature_row(X)
    shocked = apply_shocks_to_row(row, shocks)
    preds = {}
    for t in scalers.keys():
        ss = scalers[t]
        mdl = models[t][use]
        Xs = ss.transform(shocked.to_frame().T)
        preds[t] = float(mdl.predict(Xs)[0])
    return pd.Series(preds, name="pred_next_day_return")

# Example usage (edit shocks as needed)
example_shocks = {"oil": +0.05, "vix": +0.10, "gold": -0.05}  # +5% oil, +10% VIX, -5% gold
predict_with_shocks(models, scalers, X, example_shocks, use="rf")



## 9) Visualization helpers
Quick diagnostics for recent actual vs. predicted (out-of-sample via rolling last fold approximation).


In [ ]:

def plot_last_n(y: pd.Series, yhat: np.ndarray, title: str, n=120):
    plt.figure()
    y.tail(n).plot(label="Actual")
    pd.Series(yhat, index=y.index[-len(yhat):]).tail(n).plot(label="Pred (approx OOS)")
    plt.legend()
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Daily Return")
    plt.show()



## 10) Next steps & ideas
- Swap/augment models (LightGBM/XGBoost, Lasso for sparse selection, simple ARIMAX for baselines).
- Add **release calendars** for macro surprises (e.g., CPI surprise vs. consensus) for cleaner event effects.
- Add **text embeddings** and train a small classifier to map headlines → event types (CEO change, outage, pandemic scare).
- Implement **local projections/event study** around dated events to estimate impulse responses.
- Layer a simple **portfolio** rule: trade when predicted return > threshold; track backtest stats.
